# Enhanced Pixel-Level Satellite Land Cover Classification

**Unsupervised Land Cover Classification for Sentinel-2 L2A Data**

- **Training Areas**: Jaipur-Ajmer & Bikaner (Rajasthan) — 20 grid tiles
- **Validation Area**: Chandrapur (Maharashtra) — 10 grid tiles
- **Approach**: Unsupervised pixel-level classification (SLIC + KMeans → RF + CNN + U-Net)
- **Classes**: 7 land cover types
- **Output**: Multi-colored pixel-wise classified GeoTIFFs + report-ready figures

In [ ]:
# ============================================================
# CELL 1: Environment Setup
# ============================================================
import warnings, os, time, pickle, random, gc
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
import cv2
from tqdm.notebook import tqdm

warnings.filterwarnings('ignore')
plt.style.use('seaborn-v0_8-whitegrid')
RANDOM_STATE = 42
random.seed(RANDOM_STATE)
np.random.seed(RANDOM_STATE)

# --- Sklearn ---
from sklearn.ensemble import RandomForestClassifier
from sklearn.cluster import KMeans
from sklearn.mixture import GaussianMixture
from sklearn.model_selection import train_test_split
from sklearn.metrics import (classification_report, confusion_matrix,
                              accuracy_score, silhouette_score)
from sklearn.preprocessing import StandardScaler

# --- Image processing ---
from skimage import exposure, filters
from skimage.segmentation import slic, mark_boundaries
from skimage.feature import graycomatrix, graycoprops, local_binary_pattern

# --- TensorFlow ---
try:
    import tensorflow as tf
    from tensorflow.keras import layers, Model
    from tensorflow.keras.layers import (
        Input, Conv2D, MaxPooling2D, UpSampling2D, Dropout,
        BatchNormalization, Dense, Flatten, GlobalAveragePooling2D,
        concatenate)
    from tensorflow.keras.optimizers import Adam
    from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau
    from tensorflow.keras.utils import to_categorical
    tf.random.set_seed(RANDOM_STATE)
    gpus = tf.config.experimental.list_physical_devices('GPU')
    for g in gpus:
        tf.config.experimental.set_memory_growth(g, True)
    TF_OK = True
    print(f"✅ TensorFlow {tf.__version__}")
except Exception as e:
    TF_OK = False
    print(f"⚠️  TensorFlow unavailable: {e}")

# --- Rasterio ---
try:
    import rasterio
    from rasterio.enums import ColorInterp
    RASTERIO_OK = True
    print("✅ Rasterio loaded")
except ImportError:
    RASTERIO_OK = False
    print("⚠️  Rasterio unavailable — will use OpenCV fallback")

# ---- Directory setup (auto-detect project root) ----
CWD = Path.cwd()
ROOT = CWD.parent if CWD.name in ('notebooks', 'src') else CWD

DIRS = {
    'training':        ROOT / 'data' / 'training_grids',
    'validation':      ROOT / 'data' / 'validation_grids',
    'models':          ROOT / 'models' / 'saved_models',
    'classified_tifs': ROOT / 'outputs' / 'results' / 'classified_tifs',
    'visualizations':  ROOT / 'outputs' / 'visualizations',
    'reports':         ROOT / 'outputs' / 'reports',
}
for d in DIRS.values():
    d.mkdir(parents=True, exist_ok=True)

# ---- Land cover colour scheme ----
LC = {
    0: dict(name='Hills/Rocky',  color=(139,  69,  19)),
    1: dict(name='Crop Fields',  color=( 34, 139,  34)),
    2: dict(name='Fallow Land',  color=(255, 215,   0)),
    3: dict(name='Water Body',   color=(  0,   0, 255)),
    4: dict(name='Sandy River',  color=(255, 165,   0)),
    5: dict(name='Plantation',   color=(  0, 100,   0)),
    6: dict(name='Built-up',     color=(255,   0,   0)),
}
N_CLASSES = len(LC)
CMAP = np.array([LC[i]['color'] for i in range(N_CLASSES)], dtype=np.uint8)  # (7,3)

def colorize(label_map):
    """Convert integer label map → RGB image using LC colour table."""
    lm = np.clip(label_map.astype(int), 0, N_CLASSES-1)
    return CMAP[lm]

# ---- Scan files ----
EXTS = ('*.tif', '*.TIF', '*.tiff', '*.TIFF')
train_files = sorted([f for e in EXTS for f in DIRS['training'].glob(e)])
val_files   = sorted([f for e in EXTS for f in DIRS['validation'].glob(e)])
print(f"\n📁 Training files   : {len(train_files)}")
print(f"📁 Validation files : {len(val_files)}")
print("\n🎨 Land-cover scheme:")
for i, info in LC.items():
    print(f"   Class {i}: {info['name']:12s}  RGB{info['color']}")

In [ ]:
# ============================================================
# CELL 2: Image Loading & Normalisation
# ============================================================

TARGET = (512, 512)   # all images are resized to this

def load_tif(path, target=TARGET):
    """Load a single GeoTIFF → normalised float32 array (H,W),
    preserving the rasterio profile for georeferenced output."""
    path = Path(path)
    profile = None
    try:
        if RASTERIO_OK:
            with rasterio.open(path) as src:
                profile = src.profile.copy()
                img = src.read(1).astype(np.float32)
        else:
            img = cv2.imread(str(path), cv2.IMREAD_UNCHANGED).astype(np.float32)
    except Exception as e:
        print(f"  ⚠️  Could not read {path.name}: {e}")
        return None, None

    # Handle NaN / Inf
    img = np.nan_to_num(img, nan=0.0, posinf=0.0, neginf=0.0)

    # Normalise to [0, 1]
    lo, hi = np.percentile(img, 2), np.percentile(img, 98)
    if hi - lo < 1e-6:
        img = np.zeros_like(img)
    else:
        img = np.clip((img - lo) / (hi - lo), 0.0, 1.0)

    # Resize
    img = cv2.resize(img, target, interpolation=cv2.INTER_LANCZOS4)
    return img, profile


def load_dataset(file_list, desc='Loading'):
    imgs, profs = [], []
    for fp in tqdm(file_list, desc=desc):
        img, prof = load_tif(fp)
        if img is not None:
            imgs.append(img)
            profs.append(prof)
    return imgs, profs


print("Loading training images …")
train_imgs, train_profs = load_dataset(train_files, 'Training')
print(f"  ✅ Loaded {len(train_imgs)} training images")

print("Loading validation images …")
val_imgs, val_profs = load_dataset(val_files, 'Validation')
print(f"  ✅ Loaded {len(val_imgs)} validation images")

# Quick sanity plot
n = min(4, len(train_imgs))
fig, axes = plt.subplots(1, n, figsize=(4*n, 3))
if n == 1: axes = [axes]
for ax, img, fp in zip(axes, train_imgs[:n], train_files[:n]):
    ax.imshow(img, cmap='gray', vmin=0, vmax=1)
    ax.set_title(fp.name, fontsize=8)
    ax.axis('off')
plt.suptitle('Sample Training Images (normalised)', fontweight='bold')
plt.tight_layout()
plt.savefig(DIRS['visualizations'] / 'sample_training_images.png', dpi=150, bbox_inches='tight')
plt.show()
print("✅ Sample training images saved.")

In [ ]:
# ============================================================
# CELL 3: Feature Extraction & Pseudo-label Generation
#         (SLIC superpixels + KMeans cluster labels)
# ============================================================

def extract_pixel_features(img):
    """Return (H*W, n_features) feature matrix for one grayscale image."""
    h, w = img.shape
    feats = [img.ravel()]                    # raw intensity

    # Gradient magnitude
    gx = cv2.Sobel(img, cv2.CV_64F, 1, 0, ksize=3)
    gy = cv2.Sobel(img, cv2.CV_64F, 0, 1, ksize=3)
    feats.append(np.hypot(gx, gy).ravel())

    # LBP texture
    feats.append(local_binary_pattern(img, P=8, R=1, method='uniform').ravel())

    # Local mean & std at 3 window sizes
    for k in (5, 9, 15):
        kernel = np.ones((k, k), np.float32) / k**2
        lmean = cv2.filter2D(img, -1, kernel)
        lvar  = cv2.filter2D(img**2, -1, kernel) - lmean**2
        feats.extend([lmean.ravel(), np.sqrt(np.maximum(lvar, 0)).ravel()])

    # GLCM summary (contrast, homogeneity, energy) — computed on downsampled patch for speed
    thumb = cv2.resize(img, (64, 64))
    thumb8 = (thumb * 31).astype(np.uint8)
    glcm = graycomatrix(thumb8, distances=[1], angles=[0, 45], levels=32,
                        symmetric=True, normed=True)
    for prop in ('contrast', 'homogeneity', 'energy'):
        val = float(graycoprops(glcm, prop).mean())
        feats.append(np.full(h * w, val))

    return np.column_stack(feats).astype(np.float32)


def generate_pseudo_labels(img, n_classes=N_CLASSES):
    """SLIC superpixels → KMeans cluster labels → pixel-wise label map."""
    # SLIC segmentation
    img_c = np.clip(img, 0.0, 1.0)
    segs = slic(img_c, n_segments=400, compactness=8, sigma=1.0,
                start_label=0, channel_axis=None)

    seg_ids = np.unique(segs)

    # Per-segment statistics
    feat_rows, valid_ids = [], []
    for sid in seg_ids:
        px = img[segs == sid]
        if px.size < 20:
            continue
        row = [
            px.mean(), px.std(), np.median(px),
            np.percentile(px, 25), np.percentile(px, 75),
            px.min(), px.max(), px.size
        ]
        feat_rows.append(row)
        valid_ids.append(sid)

    if len(feat_rows) < n_classes:
        return (segs % n_classes).astype(np.uint8)

    X = StandardScaler().fit_transform(np.array(feat_rows))
    km = KMeans(n_clusters=n_classes, random_state=RANDOM_STATE, n_init=10)
    cluster_labels = km.fit_predict(X)

    label_map = np.zeros_like(segs, dtype=np.uint8)
    for i, sid in enumerate(valid_ids):
        label_map[segs == sid] = cluster_labels[i]

    return label_map


# ---------- Generate pseudo-labels for all training images ----------
print("Generating pseudo-labels for training images …")
t0 = time.time()
train_labels = []
for img in tqdm(train_imgs, desc='Pseudo-labelling'):
    train_labels.append(generate_pseudo_labels(img))
print(f"  ✅ Done in {time.time()-t0:.1f}s")

# ---------- Visualise a few ----------
n = min(4, len(train_imgs))
fig, axes = plt.subplots(2, n, figsize=(4*n, 7))
legend_patches = [mpatches.Patch(color=np.array(LC[i]['color'])/255,
                                  label=f"{i}: {LC[i]['name']}")
                  for i in range(N_CLASSES)]
for j in range(n):
    axes[0, j].imshow(train_imgs[j], cmap='gray')
    axes[0, j].set_title(f'Image {j+1}', fontsize=9)
    axes[0, j].axis('off')
    rgb = colorize(train_labels[j])
    axes[1, j].imshow(rgb)
    axes[1, j].set_title(f'Pseudo-labels {j+1}', fontsize=9)
    axes[1, j].axis('off')

fig.legend(handles=legend_patches, loc='lower center', ncol=N_CLASSES,
           fontsize=8, bbox_to_anchor=(0.5, -0.02))
plt.suptitle('Training Images & SLIC+KMeans Pseudo-labels', fontweight='bold')
plt.tight_layout()
plt.savefig(DIRS['visualizations'] / 'pseudo_labels_training.png',
            dpi=150, bbox_inches='tight')
plt.show()
print("✅ Pseudo-label visualisation saved.")

In [ ]:
# ============================================================
# CELL 4: Random Forest Pixel Classifier
# ============================================================

SUBSAMPLE = 8000   # pixels per image for RF training

def build_rf_dataset(imgs, labels, subsample=SUBSAMPLE):
    X_parts, y_parts = [], []
    for img, lbl in zip(imgs, labels):
        F = extract_pixel_features(img)   # (H*W, n_feat)
        L = lbl.ravel()
        n = min(F.shape[0], subsample)
        idx = np.random.choice(F.shape[0], n, replace=False)
        X_parts.append(F[idx])
        y_parts.append(L[idx])
    return np.vstack(X_parts), np.concatenate(y_parts)


print("Building RF training dataset …")
t0 = time.time()
X_rf, y_rf = build_rf_dataset(train_imgs, train_labels)
print(f"  Dataset shape: {X_rf.shape}  time: {time.time()-t0:.1f}s")

scaler_rf = StandardScaler().fit(X_rf)
X_rf_s = scaler_rf.transform(X_rf)

print("Training Random Forest …")
t0 = time.time()
rf = RandomForestClassifier(
    n_estimators=120, max_depth=18, min_samples_leaf=4,
    class_weight='balanced', random_state=RANDOM_STATE, n_jobs=-1)
rf.fit(X_rf_s, y_rf)
oob_score = rf.oob_score_ if hasattr(rf, 'oob_score_') else None
print(f"  ✅ RF trained in {time.time()-t0:.1f}s")


def rf_predict_image(img):
    F = extract_pixel_features(img)
    F_s = scaler_rf.transform(F)
    return rf.predict(F_s).reshape(img.shape).astype(np.uint8)


# Predict on validation set
print("Predicting on validation images …")
rf_preds = []
for img in tqdm(val_imgs, desc='RF predict'):
    rf_preds.append(rf_predict_image(img))
print(f"  ✅ {len(rf_preds)} prediction maps generated")

# ---------- Visualise ----------
n = min(4, len(val_imgs))
fig, axes = plt.subplots(2, n, figsize=(4*n, 7))
for j in range(n):
    axes[0, j].imshow(val_imgs[j], cmap='gray')
    axes[0, j].set_title(f'Val img {j+1}', fontsize=9)
    axes[0, j].axis('off')
    axes[1, j].imshow(colorize(rf_preds[j]))
    axes[1, j].set_title(f'RF prediction', fontsize=9)
    axes[1, j].axis('off')
legend_patches = [mpatches.Patch(color=np.array(LC[i]['color'])/255,
                                  label=f"{i}: {LC[i]['name']}")
                  for i in range(N_CLASSES)]
fig.legend(handles=legend_patches, loc='lower center', ncol=N_CLASSES,
           fontsize=8, bbox_to_anchor=(0.5, -0.02))
plt.suptitle('Random Forest — Validation Predictions', fontweight='bold')
plt.tight_layout()
plt.savefig(DIRS['visualizations'] / 'rf_predictions.png',
            dpi=150, bbox_inches='tight')
plt.show()
print("✅ RF prediction figure saved.")

In [ ]:
# ============================================================
# CELL 5: CNN Patch Classifier
# ============================================================
PATCH = 32
HALF  = PATCH // 2
STRIDE = 16
PATCHES_PER_IMG = 3000
CNN_EPOCHS = 12
BATCH = 64

def build_cnn():
    inp = Input(shape=(PATCH, PATCH, 1))
    x = Conv2D(32, 3, activation='relu', padding='same')(inp)
    x = BatchNormalization()(x)
    x = MaxPooling2D(2)(x)
    x = Dropout(0.25)(x)
    x = Conv2D(64, 3, activation='relu', padding='same')(x)
    x = BatchNormalization()(x)
    x = MaxPooling2D(2)(x)
    x = Dropout(0.25)(x)
    x = Conv2D(128, 3, activation='relu', padding='same')(x)
    x = GlobalAveragePooling2D()(x)
    x = Dropout(0.40)(x)
    x = Dense(128, activation='relu')(x)
    x = Dropout(0.40)(x)
    out = Dense(N_CLASSES, activation='softmax')(x)
    return Model(inp, out)


def extract_patches(img, label_map, n=PATCHES_PER_IMG):
    H, W = img.shape
    ys = np.random.randint(HALF, H - HALF, n)
    xs = np.random.randint(HALF, W - HALF, n)
    patches, lbls = [], []
    for y, x in zip(ys, xs):
        p = img[y-HALF:y+HALF, x-HALF:x+HALF]
        if p.shape == (PATCH, PATCH):
            patches.append(p[..., None])
            lbls.append(int(label_map[y, x]))
    return (np.array(patches, dtype=np.float32),
            np.array(lbls,   dtype=np.int32))


cnn_preds   = []
cnn_history = None

if TF_OK:
    print("Building CNN patch dataset …")
    all_p, all_l = [], []
    for img, lbl in zip(train_imgs, train_labels):
        p, l = extract_patches(img, lbl)
        all_p.append(p)
        all_l.append(l)

    Xc = np.concatenate(all_p, axis=0).astype(np.float32)
    yc = np.concatenate(all_l, axis=0).astype(np.int32)   # integer labels, NOT one-hot

    Xtr, Xv, ytr, yv = train_test_split(
        Xc, yc, test_size=0.15, random_state=RANDOM_STATE)

    # Convert to tf.Tensor so Keras 3 never sees a Python/numpy string dtype
    Xtr = tf.constant(Xtr, dtype=tf.float32)
    Xv  = tf.constant(Xv,  dtype=tf.float32)
    ytr = tf.constant(ytr, dtype=tf.int32)
    yv  = tf.constant(yv,  dtype=tf.int32)

    print(f"  Patches: {Xtr.shape[0]} train / {Xv.shape[0]} val")

    cnn = build_cnn()

    # Use explicit loss/metric objects — never strings — for Keras 3 compatibility
    cnn.compile(
        optimizer=Adam(learning_rate=1e-3),
        loss=tf.keras.losses.SparseCategoricalCrossentropy(),
        metrics=[tf.keras.metrics.SparseCategoricalAccuracy(name='accuracy')]
    )

    cbs = [
        EarlyStopping(monitor='val_accuracy', patience=5,
                      restore_best_weights=True, verbose=0),
        ReduceLROnPlateau(monitor='val_loss', factor=0.5,
                          patience=3, verbose=0)
    ]

    print(f"Training CNN ({CNN_EPOCHS} epochs) …")
    t0 = time.time()
    cnn_history = cnn.fit(
        Xtr, ytr,
        validation_data=(Xv, yv),
        epochs=CNN_EPOCHS,
        batch_size=BATCH,
        callbacks=cbs,
        verbose=1
    )
    print(f"  ✅ CNN trained in {time.time()-t0:.1f}s")

    # Plot training curves
    fig, axes = plt.subplots(1, 2, figsize=(10, 4))
    for ax, keys, title in zip(
            axes,
            [('loss', 'val_loss'), ('accuracy', 'val_accuracy')],
            ['Loss', 'Accuracy']):
        for key, label in zip(keys, ['Train', 'Val']):
            if key in cnn_history.history:
                ax.plot(cnn_history.history[key], label=label)
        ax.set_title(f'CNN {title}')
        ax.legend()
        ax.set_xlabel('Epoch')
    plt.tight_layout()
    plt.savefig(DIRS['visualizations'] / 'cnn_training_curves.png',
                dpi=150, bbox_inches='tight')
    plt.show()

    # Predict on validation
    print("CNN predicting on validation images …")
    for img in tqdm(val_imgs, desc='CNN predict'):
        H, W = img.shape
        pred = np.zeros((H, W), dtype=np.uint8)
        batch_p, batch_pos = [], []
        for y in range(HALF, H - HALF, STRIDE):
            for x in range(HALF, W - HALF, STRIDE):
                p = img[y-HALF:y+HALF, x-HALF:x+HALF]
                if p.shape == (PATCH, PATCH):
                    batch_p.append(p[..., None].astype(np.float32))
                    batch_pos.append((y, x))
        if batch_p:
            bp    = tf.constant(np.array(batch_p, dtype=np.float32))
            probs = cnn.predict(bp, batch_size=256, verbose=0)
            cls   = probs.argmax(axis=1)
            for (y, x), c in zip(batch_pos, cls):
                y0 = max(0, y - STRIDE//2); y1 = min(H, y + STRIDE//2)
                x0 = max(0, x - STRIDE//2); x1 = min(W, x + STRIDE//2)
                pred[y0:y1, x0:x1] = c
        cnn_preds.append(pred)
    print(f"  ✅ {len(cnn_preds)} CNN prediction maps")

else:
    print("⚠️  TensorFlow not available — CNN step skipped.")

In [ ]:
# ============================================================
# CELL 6: U-Net — Scratch + Pretrained Encoder (ResNet34)
# ============================================================
UNET_EPOCHS = 15
UNET_BATCH  = 4
unet_preds            = []
unet_pretrained_preds = []

# ── segmentation_models compatibility patch for Keras 3 ─────
import os
os.environ['SM_FRAMEWORK'] = 'tf.keras'

import keras.utils
if not hasattr(keras.utils, 'generic_utils'):
    import types
    keras.utils.generic_utils = types.SimpleNamespace(
        get_custom_objects=keras.utils.get_custom_objects
    )

try:
    import segmentation_models as sm
    SM_OK = True
    print("✅ segmentation_models loaded")
except Exception as e:
    SM_OK = False
    print(f"⚠️  segmentation_models failed: {e}")
    print("   Pretrained U-Net will be skipped — scratch U-Net still runs ✅")


# ════════════════════════════════════════════════════════════
# PART A — Vanilla U-Net (from scratch)
# ════════════════════════════════════════════════════════════
def build_unet_scratch(input_shape=(512, 512, 1), n_classes=N_CLASSES):

    from tensorflow.keras.layers import Conv2DTranspose

    def enc_block(x, f, drop=0.1):
        x = Conv2D(f, 3, activation='relu', padding='same')(x)
        x = BatchNormalization()(x)
        x = Conv2D(f, 3, activation='relu', padding='same')(x)
        x = BatchNormalization()(x)
        p = MaxPooling2D(2)(x)
        p = Dropout(drop)(p)
        return x, p

    def dec_block(x, skip, f, drop=0.1):
        x = Conv2DTranspose(f, kernel_size=2, strides=2, padding='same')(x)
        x = concatenate([x, skip])
        x = Conv2D(f, 3, activation='relu', padding='same')(x)
        x = BatchNormalization()(x)
        x = Conv2D(f, 3, activation='relu', padding='same')(x)
        x = BatchNormalization()(x)
        x = Dropout(drop)(x)
        return x

    inp = Input(input_shape)
    s1, p1 = enc_block(inp, 32,  0.10)
    s2, p2 = enc_block(p1,  64,  0.15)
    s3, p3 = enc_block(p2,  128, 0.20)
    s4, p4 = enc_block(p3,  256, 0.25)

    b = Conv2D(512, 3, activation='relu', padding='same')(p4)
    b = BatchNormalization()(b)
    b = Dropout(0.40)(b)

    d4 = dec_block(b,  s4, 256, 0.25)
    d3 = dec_block(d4, s3, 128, 0.20)
    d2 = dec_block(d3, s2,  64, 0.15)
    d1 = dec_block(d2, s1,  32, 0.10)

    out = Conv2D(n_classes, 1, activation='softmax')(d1)
    return Model(inp, out)


if TF_OK:
    # ── Prepare integer-label data ───────────────────────────
    imgs_np = np.array(train_imgs, dtype=np.float32)[..., None]  # (N,512,512,1)
    lbls_np = np.array(train_labels, dtype=np.int32)              # (N,512,512)

    Xtu, Xvu, ytu, yvu = train_test_split(
        imgs_np, lbls_np, test_size=0.15, random_state=RANDOM_STATE)

    Xtu_t = tf.constant(Xtu, dtype=tf.float32)
    Xvu_t = tf.constant(Xvu, dtype=tf.float32)
    ytu_t = tf.constant(ytu, dtype=tf.int32)
    yvu_t = tf.constant(yvu, dtype=tf.int32)

    print(f"\n{'='*55}")
    print("PART A — Vanilla U-Net (from scratch)")
    print(f"{'='*55}")
    print(f"Train: {Xtu_t.shape[0]} | Val: {Xvu_t.shape[0]}")

    unet_scratch = build_unet_scratch()
    unet_scratch.compile(
        optimizer=Adam(learning_rate=1e-4),
        loss=tf.keras.losses.SparseCategoricalCrossentropy(),
        metrics=[tf.keras.metrics.SparseCategoricalAccuracy(name='accuracy')]
    )
    print(f"Parameters: {unet_scratch.count_params():,}")

    cbs_scratch = [
        EarlyStopping(monitor='val_loss', patience=6,
                      restore_best_weights=True, verbose=1),
        ReduceLROnPlateau(monitor='val_loss', factor=0.5,
                          patience=3, verbose=1)
    ]

    print(f"Training ({UNET_EPOCHS} epochs, batch={UNET_BATCH}) …")
    t0 = time.time()
    hist_scratch = unet_scratch.fit(
        Xtu_t, ytu_t,
        validation_data=(Xvu_t, yvu_t),
        epochs=UNET_EPOCHS,
        batch_size=UNET_BATCH,
        callbacks=cbs_scratch,
        verbose=1
    )
    print(f"✅ Scratch U-Net trained in {time.time()-t0:.1f}s")

    # Predict
    val_np_1ch = tf.constant(
        np.array(val_imgs, dtype=np.float32)[..., None], dtype=tf.float32)
    probs_scratch = unet_scratch.predict(val_np_1ch, batch_size=2, verbose=1)
    unet_preds = [probs_scratch[i].argmax(axis=-1).astype(np.uint8)
                  for i in range(len(probs_scratch))]
    print(f"✅ {len(unet_preds)} scratch U-Net prediction maps")

    # Plot scratch training curves
    fig, axes = plt.subplots(1, 2, figsize=(10, 4))
    for ax, keys, title in zip(
            axes,
            [('loss', 'val_loss'), ('accuracy', 'val_accuracy')],
            ['Loss', 'Accuracy']):
        for key, label in zip(keys, ['Train', 'Val']):
            if key in hist_scratch.history:
                ax.plot(hist_scratch.history[key], label=label)
        ax.set_title(f'U-Net Scratch — {title}')
        ax.legend()
        ax.set_xlabel('Epoch')
    plt.suptitle('Vanilla U-Net Training Curves', fontweight='bold')
    plt.tight_layout()
    plt.savefig(DIRS['visualizations'] / 'unet_scratch_curves.png',
                dpi=150, bbox_inches='tight')
    plt.show()


# ════════════════════════════════════════════════════════════
# PART B — Pretrained Encoder U-Net (ResNet34 + ImageNet)
# ════════════════════════════════════════════════════════════
    if SM_OK:
        print(f"\n{'='*55}")
        print("PART B — Pretrained U-Net (ResNet34, ImageNet weights)")
        print(f"{'='*55}")
        print("""
  HOW IT WORKS:
  ┌────────────────────────────────────────────────┐
  │  ResNet34 ENCODER  (weights from ImageNet)     │
  │  • already knows edges, textures, shapes       │
  │  • 1.2M images worth of visual knowledge       │
  ├────────────────────────────────────────────────┤
  │  U-Net DECODER     (trained on your 20 tiles)  │
  │  • learns to map ResNet features → land cover  │
  │  • far fewer parameters to learn from scratch  │
  └────────────────────────────────────────────────┘
        """)

        # segmentation_models needs 3-channel input
        # Repeat grayscale channel 3× to simulate RGB
        imgs_3ch = np.repeat(
            np.array(train_imgs, dtype=np.float32)[..., None],
            3, axis=-1)                                           # (N,512,512,3)

        Xtu3, Xvu3, ytu3, yvu3 = train_test_split(
            imgs_3ch, lbls_np, test_size=0.15, random_state=RANDOM_STATE)

        Xtu3_t = tf.constant(Xtu3, dtype=tf.float32)
        Xvu3_t = tf.constant(Xvu3, dtype=tf.float32)
        ytu3_t = tf.constant(ytu3, dtype=tf.int32)
        yvu3_t = tf.constant(yvu3, dtype=tf.int32)

        unet_pretrained = sm.Unet(
            backbone_name='resnet34',
            input_shape=(512, 512, 3),
            classes=N_CLASSES,
            activation='softmax',
            encoder_weights='imagenet',
            encoder_freeze=False
        )
        unet_pretrained.compile(
            optimizer=Adam(learning_rate=5e-5),
            loss=tf.keras.losses.SparseCategoricalCrossentropy(),
            metrics=[tf.keras.metrics.SparseCategoricalAccuracy(name='accuracy')]
        )
        print(f"Parameters: {unet_pretrained.count_params():,}")
        print("Encoder initialised from ImageNet — not random!")

        cbs_pt = [
            EarlyStopping(monitor='val_loss', patience=6,
                          restore_best_weights=True, verbose=1),
            ReduceLROnPlateau(monitor='val_loss', factor=0.5,
                              patience=3, verbose=1)
        ]

        print(f"Training ({UNET_EPOCHS} epochs, batch={UNET_BATCH}) …")
        t0 = time.time()
        hist_pretrained = unet_pretrained.fit(
            Xtu3_t, ytu3_t,
            validation_data=(Xvu3_t, yvu3_t),
            epochs=UNET_EPOCHS,
            batch_size=UNET_BATCH,
            callbacks=cbs_pt,
            verbose=1
        )
        print(f"✅ Pretrained U-Net trained in {time.time()-t0:.1f}s")

        # Predict
        val_np_3ch = tf.constant(
            np.repeat(
                np.array(val_imgs, dtype=np.float32)[..., None],
                3, axis=-1),
            dtype=tf.float32)
        probs_pt = unet_pretrained.predict(val_np_3ch, batch_size=2, verbose=1)
        unet_pretrained_preds = [
            probs_pt[i].argmax(axis=-1).astype(np.uint8)
            for i in range(len(probs_pt))
        ]
        print(f"✅ {len(unet_pretrained_preds)} pretrained U-Net prediction maps")

        # Comparison training curves
        fig, axes = plt.subplots(1, 2, figsize=(14, 5))
        for ax, keys, title in zip(
                axes,
                [('loss', 'val_loss'), ('accuracy', 'val_accuracy')],
                ['Loss', 'Accuracy']):
            for hist, lbl, ls in [
                    (hist_scratch,    'Scratch (random init)', '-'),
                    (hist_pretrained, 'Pretrained (ImageNet)', '--')]:
                for key, split in zip(keys, ['Train', 'Val']):
                    if key in hist.history:
                        ax.plot(hist.history[key],
                                label=f'{lbl} {split}', linestyle=ls)
            ax.set_title(f'Scratch vs Pretrained — {title}', fontweight='bold')
            ax.legend(fontsize=8)
            ax.set_xlabel('Epoch')
        plt.suptitle('Transfer Learning Effect: Scratch vs ImageNet-Pretrained ResNet34',
                     fontweight='bold')
        plt.tight_layout()
        plt.savefig(DIRS['visualizations'] / 'unet_comparison_curves.png',
                    dpi=150, bbox_inches='tight')
        plt.show()
        print("✅ Comparison curves saved.")

    else:
        print("\n⚠️  segmentation_models not available — pretrained step skipped.")
        print("   Scratch U-Net still ran successfully ✅")
        unet_pretrained_preds = []

else:
    print("⚠️  TensorFlow not available — U-Net steps skipped.")
    unet_pretrained_preds = []

In [ ]:
# ============================================================
# CELL 7: Ensemble & Comprehensive Visualisation
# ============================================================

from scipy.stats import mode as sc_mode

def ensemble_vote(pred_list):
    """Majority-vote across a list of uint8 label maps (same shape)."""
    stack = np.stack(pred_list, axis=0).astype(np.int32)   # (M, H, W)
    result = sc_mode(stack, axis=0, keepdims=False)
    # handle both old scipy (returns tuple) and new scipy (returns ModeResult)
    mode_arr = result.mode if hasattr(result, 'mode') else result[0]
    # squeeze out any leading dim
    mode_arr = np.squeeze(mode_arr)
    return mode_arr.astype(np.uint8)


# ── Collect all available predictions ───────────────────────
available_models = {'RF': rf_preds}
if cnn_preds:               available_models['CNN']             = cnn_preds
if unet_preds:              available_models['U-Net Scratch']   = unet_preds
if unet_pretrained_preds:   available_models['U-Net Pretrained']= unet_pretrained_preds

# ── Build ensemble from everything available ────────────────
ensemble_preds = []
for i in range(len(val_imgs)):
    preds_i = [rf_preds[i]]
    if i < len(cnn_preds):               preds_i.append(cnn_preds[i])
    if i < len(unet_preds):              preds_i.append(unet_preds[i])
    if i < len(unet_pretrained_preds):   preds_i.append(unet_pretrained_preds[i])
    if len(preds_i) == 1:
        ensemble_preds.append(preds_i[0])
    else:
        ensemble_preds.append(ensemble_vote(preds_i))

available_models['Ensemble'] = ensemble_preds

print(f"Models in ensemble: {list(available_models.keys())}")
print(f"Validation images : {len(val_imgs)}")


# ════════════════════════════════════════════════════════════
# FIGURE 1 — Side-by-side model comparison
# ════════════════════════════════════════════════════════════
n_models = len(available_models)
n_show   = min(5, len(val_imgs))

fig, axes = plt.subplots(n_show, n_models + 1,
                          figsize=(4 * (n_models + 1), 4 * n_show))
if n_show == 1:
    axes = axes[None, :]

legend_patches = [
    mpatches.Patch(color=np.array(LC[i]['color'])/255,
                   label=f"{i}: {LC[i]['name']}")
    for i in range(N_CLASSES)
]

for row in range(n_show):
    # Original image
    axes[row, 0].imshow(val_imgs[row], cmap='gray')
    axes[row, 0].set_title(f'Val {row+1}\n(original)', fontsize=8)
    axes[row, 0].axis('off')

    # Each model
    for col, (name, preds) in enumerate(available_models.items(), start=1):
        if row < len(preds):
            axes[row, col].imshow(colorize(preds[row]))
        axes[row, col].set_title(name, fontsize=8, fontweight='bold')
        axes[row, col].axis('off')

fig.legend(handles=legend_patches, loc='lower center', ncol=N_CLASSES,
           fontsize=7, bbox_to_anchor=(0.5, -0.01), framealpha=0.9)
plt.suptitle(
    'Model Comparison — Validation Predictions\n'
    'Chandrapur, Maharashtra (trained on Rajasthan)',
    fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig(DIRS['visualizations'] / 'model_comparison.png',
            dpi=150, bbox_inches='tight')
plt.show()
print("✅ Model comparison figure saved.")


# ════════════════════════════════════════════════════════════
# FIGURE 2 — Scratch vs Pretrained U-Net direct comparison
#            (only shown if both are available)
# ════════════════════════════════════════════════════════════
if unet_preds and unet_pretrained_preds:
    n_show2 = min(4, len(val_imgs))
    fig2, axes2 = plt.subplots(n_show2, 3,
                                figsize=(12, 4 * n_show2))
    if n_show2 == 1:
        axes2 = axes2[None, :]

    for row in range(n_show2):
        axes2[row, 0].imshow(val_imgs[row], cmap='gray')
        axes2[row, 0].set_title(f'Original (val {row+1})', fontsize=9)
        axes2[row, 0].axis('off')

        axes2[row, 1].imshow(colorize(unet_preds[row]))
        axes2[row, 1].set_title('U-Net Scratch\n(random init)', fontsize=9)
        axes2[row, 1].axis('off')

        axes2[row, 2].imshow(colorize(unet_pretrained_preds[row]))
        axes2[row, 2].set_title('U-Net Pretrained\n(ResNet34 / ImageNet)', fontsize=9)
        axes2[row, 2].axis('off')

    fig2.legend(handles=legend_patches, loc='lower center', ncol=N_CLASSES,
                fontsize=7, bbox_to_anchor=(0.5, -0.01), framealpha=0.9)
    plt.suptitle(
        'Transfer Learning Effect\n'
        'Scratch U-Net vs ImageNet-Pretrained ResNet34 Encoder',
        fontsize=12, fontweight='bold')
    plt.tight_layout()
    plt.savefig(DIRS['visualizations'] / 'unet_scratch_vs_pretrained.png',
                dpi=150, bbox_inches='tight')
    plt.show()
    print("✅ Scratch vs pretrained comparison figure saved.")


# ════════════════════════════════════════════════════════════
# FIGURE 3 — Class distribution bar chart (all models)
# ════════════════════════════════════════════════════════════
def class_distribution(preds_list):
    all_px = np.concatenate([p.ravel() for p in preds_list])
    total  = all_px.size
    return {LC[c]['name']: round(np.sum(all_px == c) / total * 100, 2)
            for c in range(N_CLASSES)}

dist_data = {name: class_distribution(preds[:len(val_imgs)])
             for name, preds in available_models.items()}

df_dist = pd.DataFrame(dist_data)

ax = df_dist.T.plot(kind='bar', figsize=(14, 5), width=0.75,
                     color=[np.array(LC[i]['color'])/255
                            for i in range(N_CLASSES)])
ax.set_ylabel('% pixels', fontsize=11)
ax.set_xlabel('Model', fontsize=11)
ax.set_title('Predicted Class Distribution Across Validation Images',
             fontweight='bold', fontsize=12)
ax.legend(loc='upper right', fontsize=8, title='Land Cover Class')
plt.xticks(rotation=20, ha='right')
plt.tight_layout()
plt.savefig(DIRS['visualizations'] / 'class_distribution.png',
            dpi=150, bbox_inches='tight')
plt.show()
print("✅ Class distribution chart saved.")
print()
print(df_dist.to_string())


# ════════════════════════════════════════════════════════════
# FIGURE 4 — Cross-model pixel agreement heatmap
# ════════════════════════════════════════════════════════════
model_names = list(available_models.keys())
n_m = len(model_names)
agree_matrix = np.zeros((n_m, n_m))

for i, ma in enumerate(model_names):
    for j, mb in enumerate(model_names):
        pa = available_models[ma][:len(val_imgs)]
        pb = available_models[mb][:len(val_imgs)]
        agreement = np.mean([np.mean(a == b) for a, b in zip(pa, pb)]) * 100
        agree_matrix[i, j] = agreement

fig4, ax4 = plt.subplots(figsize=(8, 6))
im = ax4.imshow(agree_matrix, cmap='YlGn', vmin=0, vmax=100)
plt.colorbar(im, ax=ax4, label='Pixel Agreement (%)')
ax4.set_xticks(range(n_m)); ax4.set_xticklabels(model_names, rotation=30, ha='right')
ax4.set_yticks(range(n_m)); ax4.set_yticklabels(model_names)
for i in range(n_m):
    for j in range(n_m):
        ax4.text(j, i, f'{agree_matrix[i,j]:.1f}%',
                 ha='center', va='center', fontsize=9,
                 color='black' if agree_matrix[i,j] < 70 else 'white')
ax4.set_title('Cross-Model Pixel Agreement (%)', fontweight='bold', fontsize=12)
plt.tight_layout()
plt.savefig(DIRS['visualizations'] / 'cross_model_agreement.png',
            dpi=150, bbox_inches='tight')
plt.show()
print("✅ Cross-model agreement heatmap saved.")


# ════════════════════════════════════════════════════════════
# FIGURE 5 — Ensemble confidence map
#            (how many models agreed on the winning class)
# ════════════════════════════════════════════════════════════
HERO = 0   # which validation image to show

pred_stack = []
for name, preds in available_models.items():
    if name != 'Ensemble' and HERO < len(preds):
        pred_stack.append(preds[HERO].astype(np.int32))

if len(pred_stack) > 1:
    stack      = np.stack(pred_stack, axis=0)          # (M, H, W)
    ens_cls    = ensemble_preds[HERO]                  # (H, W)
    # agreement = fraction of models that voted for the winning class
    votes_for_winner = np.sum(stack == ens_cls[None, ...], axis=0)
    confidence = votes_for_winner / len(pred_stack)    # 0–1

    fig5, axes5 = plt.subplots(1, 3, figsize=(15, 5))

    axes5[0].imshow(val_imgs[HERO], cmap='gray')
    axes5[0].set_title('Original Image', fontweight='bold')
    axes5[0].axis('off')

    axes5[1].imshow(colorize(ens_cls))
    axes5[1].set_title('Ensemble Prediction', fontweight='bold')
    axes5[1].axis('off')

    conf_im = axes5[2].imshow(confidence, cmap='RdYlGn', vmin=0, vmax=1)
    plt.colorbar(conf_im, ax=axes5[2], label='Model Agreement (0=low, 1=high)')
    axes5[2].set_title('Prediction Confidence\n(fraction of models agreeing)',
                        fontweight='bold')
    axes5[2].axis('off')

    plt.suptitle(f'Ensemble Confidence Map — Validation Image {HERO+1}',
                 fontsize=12, fontweight='bold')
    plt.tight_layout()
    plt.savefig(DIRS['visualizations'] / 'ensemble_confidence.png',
                dpi=150, bbox_inches='tight')
    plt.show()
    print("✅ Ensemble confidence map saved.")

print(f"\n✅ Cell 7 complete — {len(available_models)} models visualised.")
print(f"   Figures saved to: {DIRS['visualizations']}")

In [ ]:
# ============================================================
# CELL 8: Export Classified GeoTIFFs
# ============================================================

CLASS_COLOURS_RASTERIO = {
    i: (int(LC[i]['color'][0]),
        int(LC[i]['color'][1]),
        int(LC[i]['color'][2]),
        255)
    for i in range(N_CLASSES)
}

def save_classified_tif(label_map, ref_profile, out_path):
    """Write a uint8 label map as a palette-coloured GeoTIFF."""
    out_path = Path(out_path)
    H, W = label_map.shape
    if ref_profile is not None and RASTERIO_OK:
        prof = ref_profile.copy()
        prof.update(driver='GTiff', count=1, dtype='uint8',
                    height=H, width=W, compress='lzw', nodata=None)
        with rasterio.open(out_path, 'w', **prof) as dst:
            dst.write(label_map.astype(np.uint8), 1)
            dst.write_colormap(1, CLASS_COLOURS_RASTERIO)
    else:
        # Fallback: save RGB PNG
        rgb = colorize(label_map)
        out_png = out_path.with_suffix('.png')
        cv2.imwrite(str(out_png), cv2.cvtColor(rgb, cv2.COLOR_RGB2BGR))


print("Exporting classified GeoTIFFs …")

export_models = {'RF': rf_preds, 'Ensemble': ensemble_preds}
if cnn_preds:   export_models['CNN']   = cnn_preds
if unet_preds:  export_models['UNet']  = unet_preds

for model_name, preds in export_models.items():
    model_dir = DIRS['classified_tifs'] / model_name.lower()
    model_dir.mkdir(exist_ok=True)
    for i, (pred, fp, prof) in enumerate(zip(preds, val_files, val_profs)):
        stem  = fp.stem
        out_p = model_dir / f"{stem}_{model_name}_classified.tif"
        save_classified_tif(pred, prof, out_p)
    print(f"  ✅ {model_name}: {len(preds)} TIFs written → {model_dir}")

print("\n✅ All GeoTIFFs exported.")

In [ ]:
# ============================================================
# CELL 9: Quantitative Analysis & Report Statistics
# ============================================================

import textwrap

# ---- Self-consistency: compare RF vs Ensemble pixel agreement ----
def pixel_agreement(preds_a, preds_b):
    agrees = [np.mean(a == b) for a, b in zip(preds_a, preds_b)]
    return np.mean(agrees) * 100

agreements = {}
model_names = list(available_models.keys())
for i, ma in enumerate(model_names):
    for mb in model_names[i+1:]:
        pa = available_models[ma][:len(val_imgs)]
        pb = available_models[mb][:len(val_imgs)]
        agreements[f'{ma} vs {mb}'] = pixel_agreement(pa, pb)

# ---- Confusion matrix: RF vs ensemble (cross-model) ----
rf_flat  = np.concatenate([p.ravel() for p in rf_preds])
ens_flat = np.concatenate([p.ravel() for p in ensemble_preds])

cm = confusion_matrix(rf_flat, ens_flat,
                       labels=list(range(N_CLASSES)))
cm_norm = cm.astype(float) / cm.sum(axis=1, keepdims=True).clip(1)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
cls_names = [LC[i]['name'] for i in range(N_CLASSES)]

sns.heatmap(cm_norm, annot=True, fmt='.2f', cmap='Blues',
            xticklabels=cls_names, yticklabels=cls_names, ax=axes[0])
axes[0].set_title('Normalised CM: RF vs Ensemble')
axes[0].set_xlabel('Ensemble'); axes[0].set_ylabel('RF')
plt.setp(axes[0].get_xticklabels(), rotation=40, ha='right', fontsize=8)
plt.setp(axes[0].get_yticklabels(), rotation=0, fontsize=8)

# Agreement bar chart
bars = list(agreements.values())
lbls = list(agreements.keys())
axes[1].barh(lbls, bars, color='steelblue')
axes[1].set_xlabel('Pixel Agreement (%)')
axes[1].set_title('Cross-Model Pixel Agreement')
axes[1].set_xlim(0, 100)
for bar, val in zip(axes[1].patches, bars):
    axes[1].text(val + 0.5, bar.get_y() + bar.get_height()/2,
                 f'{val:.1f}%', va='center', fontsize=8)
plt.tight_layout()
plt.savefig(DIRS['visualizations'] / 'quantitative_analysis.png',
            dpi=150, bbox_inches='tight')
plt.show()

# ---- Print model-agreement table ----
print("\n📊 Cross-Model Pixel Agreement:")
for pair, ag in agreements.items():
    print(f"   {pair:<35}: {ag:.2f}%")

# ---- Save summary CSV ----
rows = []
for model_name, preds in available_models.items():
    all_px = np.concatenate([p.ravel() for p in preds[:len(val_imgs)]])
    for cls_id in range(N_CLASSES):
        pct = np.mean(all_px == cls_id) * 100
        rows.append({'Model': model_name, 'Class': LC[cls_id]['name'], '%pixels': round(pct, 2)})

df_out = pd.DataFrame(rows)
csv_path = DIRS['reports'] / 'class_distribution_summary.csv'
df_out.to_csv(csv_path, index=False)
print(f"\n✅ Class-distribution CSV saved: {csv_path}")
display(df_out.pivot(index='Class', columns='Model', values='%pixels').round(2))

In [ ]:
# ============================================================
# CELL 10: Final Report-Ready Summary Figure
# ============================================================

# Pick the best single validation image for the hero figure
HERO_IDX = 0

fig = plt.figure(figsize=(20, 12))
titles = ['Original Image'] + list(available_models.keys())
cols   = len(titles)

for j, title in enumerate(titles):
    ax = fig.add_subplot(2, cols, j + 1)
    if j == 0:
        ax.imshow(val_imgs[HERO_IDX], cmap='gray')
    else:
        model_name = list(available_models.keys())[j - 1]
        preds = available_models[model_name]
        ax.imshow(colorize(preds[HERO_IDX]))
    ax.set_title(title, fontweight='bold', fontsize=11)
    ax.axis('off')

# SLIC superpixel overlay in bottom row
img_c = np.clip(val_imgs[HERO_IDX], 0, 1)
segs  = slic(img_c, n_segments=200, compactness=8, sigma=1.0,
             start_label=0, channel_axis=None)
bound = mark_boundaries(img_c, segs, color=(1, 1, 0))

ax_slic = fig.add_subplot(2, cols, cols + 1)
ax_slic.imshow(bound)
ax_slic.set_title('SLIC Superpixels', fontsize=11, fontweight='bold')
ax_slic.axis('off')

# Class distribution pie for ensemble
ax_pie = fig.add_subplot(2, cols, cols + 2)
ens = ensemble_preds[HERO_IDX].ravel()
counts = [np.sum(ens == c) for c in range(N_CLASSES)]
colors_pie = [np.array(LC[c]['color'])/255 for c in range(N_CLASSES)]
wedges, texts, autotexts = ax_pie.pie(
    counts, labels=[LC[c]['name'] for c in range(N_CLASSES)],
    colors=colors_pie, autopct='%1.1f%%', startangle=90,
    textprops={'fontsize': 7})
for at in autotexts: at.set_fontsize(7)
ax_pie.set_title('Ensemble Class Distribution\n(validation img 1)',
                  fontsize=10, fontweight='bold')

# Legend
legend_patches = [mpatches.Patch(color=np.array(LC[i]['color'])/255,
                                  label=f"{i}: {LC[i]['name']}")
                  for i in range(N_CLASSES)]
fig.legend(handles=legend_patches, loc='lower center', ncol=N_CLASSES,
           fontsize=8, bbox_to_anchor=(0.5, -0.01), framealpha=0.9)

plt.suptitle(
    'Satellite Land Cover Classification — Chandrapur (Maharashtra)\n'
    'Trained on Jaipur-Ajmer & Bikaner (Rajasthan) · Sentinel-2 L2A',
    fontsize=13, fontweight='bold', y=1.01)
plt.tight_layout()
hero_path = DIRS['visualizations'] / 'hero_figure.png'
plt.savefig(hero_path, dpi=200, bbox_inches='tight')
plt.show()
print(f"✅ Hero figure saved: {hero_path}")

In [ ]:
# ============================================================
# CELL 11: Markdown Report Generation
# ============================================================

report_lines = [
    "# Pixel-Level Land Cover Classification — Report Summary",
    "",
    "## 1. Study Overview",
    "",
    f"| Parameter | Value |",
    f"|-----------|-------|",
    f"| Satellite | Sentinel-2 L2A |",
    f"| Training region | Jaipur-Ajmer & Bikaner, Rajasthan |",
    f"| Validation region | Chandrapur, Maharashtra |",
    f"| Training tiles | {len(train_imgs)} |",
    f"| Validation tiles | {len(val_imgs)} |",
    f"| Image size | {TARGET[0]} × {TARGET[1]} pixels |",
    f"| Land cover classes | {N_CLASSES} |",
    "",
    "## 2. Land Cover Classes",
    "",
    "| ID | Class | RGB Colour |",
    "|----|-------|------------|",
]
for i, info in LC.items():
    report_lines.append(f"| {i} | {info['name']} | {info['color']} |")

report_lines += [
    "",
    "## 3. Methodology",
    "",
    "### Pseudo-label Generation",
    "- SLIC superpixel segmentation (n_segments=400, compactness=8)",
    "- KMeans clustering (k=7) on per-segment statistics (mean, std, median, "
    "percentiles, GLCM contrast/homogeneity/energy)",
    "",
    "### Models",
    "",
    "**Random Forest** — pixel-level classifier trained on hand-crafted "
    "features (intensity, gradient magnitude, LBP, multi-scale local "
    "statistics, GLCM summary). 120 trees, max depth 18, balanced class weights.",
    "",
]

if TF_OK:
    report_lines += [
        "**CNN Patch Classifier** — 32×32 patch classification with 3-block "
        "encoder (32→64→128 filters) + GlobalAveragePooling + Dense head. "
        f"Trained for {CNN_EPOCHS} epochs, batch {BATCH}.",
        "",
        "**U-Net Segmentation** — 4-level encoder-decoder with skip connections. "
        f"Combined Dice + Categorical-CE loss. Trained for {UNET_EPOCHS} epochs, "
        f"batch {UNET_BATCH}.",
        "",
    ]

report_lines += [
    "**Ensemble** — majority-vote across all trained models.",
    "",
    "## 4. Cross-Model Pixel Agreement",
    "",
    "| Comparison | Agreement |",
    "|------------|----------|",
]
for pair, ag in agreements.items():
    report_lines.append(f"| {pair} | {ag:.2f}% |")

report_lines += [
    "",
    "## 5. Class Distribution Summary (%)",
    "",
    df_out.pivot(index='Class', columns='Model', values='%pixels').round(2).to_markdown(),
    "",
    "## 6. Output Artefacts",
    "",
    f"- Classified GeoTIFFs: `{DIRS['classified_tifs']}`",
    f"- Figures: `{DIRS['visualizations']}`",
    f"- CSV summary: `{csv_path}`",
]

report_path = DIRS['reports'] / 'classification_report.md'
report_path.write_text('\n'.join(report_lines), encoding='utf-8')
print(f"✅ Report written: {report_path}")
print()
print('\n'.join(report_lines[:40]))
print('\n... (see file for full report)')

In [ ]:
# ============================================================
# CELL 12: Final Execution Summary
# ============================================================

print("=" * 65)
print("   PIXEL-LEVEL SATELLITE CLASSIFICATION — COMPLETE")
print("=" * 65)
print(f"\n📊 Dataset")
print(f"   Training images   : {len(train_imgs)}")
print(f"   Validation images : {len(val_imgs)}")
print(f"   Image resolution  : {TARGET[0]}×{TARGET[1]} px")

print(f"\n🤖 Models trained")
models_run = ['Random Forest']
if cnn_preds:   models_run.append('CNN patch classifier')
if unet_preds:  models_run.append('U-Net segmentation')
models_run.append('Ensemble (majority vote)')
for m in models_run:
    print(f"   ✅ {m}")

print(f"\n📁 Outputs")
print(f"   Classified TIFs  : {DIRS['classified_tifs']}")
print(f"   Figures          : {DIRS['visualizations']}")
print(f"   Report           : {DIRS['reports'] / 'classification_report.md'}")
print(f"   CSV summary      : {csv_path}")

print(f"\n🎨 Land-cover classes used:")
for i, info in LC.items():
    print(f"   {i}  {info['name']:<14} RGB{info['color']}")

print(f"\n🏁 All done — ready for report writing!")
print("=" * 65)